## Toolset

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import cantera as ct
import majordome as mj
import numpy as np

from pathlib import Path
from ruamel.yaml import YAML

## Preliminary calculations

In [3]:
# Total flow rate [Nm³/h].
qdot_fuel = 500.0

# Molar composition of fuel [-].
X_fuel = "CH4: 1"

# Molar composition of oxidizer [-].
X_oxid = "N2: 0.79, O2: 0.21"

# Mechanism for base calculations.
mech = "chemkin/BFER_methane.yaml"
phase = "CH4_BFER_multi"

# Burner diameters [mm].
diam_fuel     = 27
diam_oxid_ext = 342
diam_oxid_int = 76
thick_wall    = 2.5

### Energy input

In [4]:
normal_flow = mj.NormalFlowRate(mech, X=X_fuel, name=phase)
mdot_fuel = normal_flow(qdot_fuel) / 4

chon = mj.CombustionAtmosphereCHON(mech, phase=phase, basis="mole")
lhv = chon.solution_heating_value(X_fuel, X_oxid)

supply = mj.CombustionPowerSupply(
    power       = lhv * mdot_fuel * 1000,
    equivalence = 1.0,
    fuel        = X_fuel,
    oxidizer    = X_oxid,
    mechanism   = mech,
    phase       = phase,
    basis       = "mole",
)

_ = supply.report(notebook=True)

| Property                  | Unit   |     Value |
|---------------------------|--------|-----------|
| Required power            | kW     | 1243.27   |
| Lower heating value       | MJ/kg  |   50.0254 |
| Fuel mass flow rate       | kg/h   |   89.4699 |
| Oxidizer mass flow rate   | kg/h   | 1532.35   |
| Total mass flow rate      | kg/h   | 1621.82   |
| Fuel volume flow rate     | Nm³/h  |  125      |
| Oxidizer volume flow rate | Nm³/h  | 1190.48   |
| Total volume flow rate    | Nm³/h  | 1315.48   |
| Water production          | kg/h   |  200.935  |
| Carbon dioxide production | kg/h   |  245.433  |
| Total emissions           | kg/h   |  446.368  |

### Geometry

In [5]:
A_fuel_pipe = np.pi * (diam_fuel / 2 + thick_wall)**2
A_fuel_tot  = 4 * A_fuel_pipe

A_air_ext = np.pi * (diam_oxid_ext / 2)**2
A_air_int = np.pi * (diam_oxid_int / 2)**2

A_air = A_air_ext - A_air_int - A_fuel_tot
A_air_one = A_air / 4

e_wall = float(0.001 * thick_wall)
D_fuel = float(0.001 * diam_fuel)
D_oxid = float(0.001 * np.sqrt(4 * (A_air_one + A_fuel_pipe) / np.pi))

In [6]:
data = {
    "wedge_angle": 2.0,
    "len_inlet_ax": 3 * D_fuel,
    "len_inlet_an": 3 * D_fuel,
    "len_disperse": 1.00,
    "len_flue_out": 2.00,
    "dia_inlet_ax": D_fuel,
    "thk_inlet_wl": e_wall,
    "thk_inlet_an": D_oxid / 2,
    "thk_side_box": 2 * D_oxid,
}

yaml = YAML()
yaml.default_flow_style = False
yaml.indent(mapping=2, sequence=4, offset=2)

output_file = Path("domain.yaml")
with output_file.open("w", encoding="utf-8") as f:
    yaml.dump(data, f)

## Kinetics data

In [7]:
sutherland = mj.SutherlandFitting(mech, name=phase)
sutherland.fit(np.linspace(400, 2300, 100))

transport = Path(mech).parent / "transport"
transport.unlink()

_ = sutherland.as_openfoam_dict(transport)

In [8]:
#| echo: false
# Use the following for inspection:
# figs = Path(mech).parent / "media"
# figs.mkdir(exist_ok=True)

# for s in coef["species"]:
#     p = sutherland.plot_species(s)
#     p.savefig(figs / f"{s}.png")

In [9]:
#| echo: false
#| tbl-cap: Sutherland fitting coefficients for transport properties.
#| label: tbl-sutherland-coefs
sutherland.coefs_table

,species,As [uPa.s],Ts [K],RMSE [uPa.s]
0,O2,1.845210,207.527886,0.528664
1,H2O,2.281734,974.848255,0.217912
2,CH4,1.076964,224.165687,0.263242
3,CO,1.569128,203.847264,0.466735
4,CO2,1.688398,296.196009,0.213426
5,N2,1.596546,203.635451,0.475951


### Conditions setup